In [14]:
import os
import re


In [15]:
# Regex patterns
CLASS_COMPONENT_REGEX = re.compile(r'class\s+(\w+)\s+extends\s+React\.(Component|PureComponent)')
FUNCTION_COMPONENT_REGEX = re.compile(r'function\s+(\w+)\s*\([^)]*\)\s*\{\s*return\s+<[^>]+>[\s\S]*?\}')
ARROW_FUNCTION_COMPONENT_REGEX = re.compile(r'const\s+(\w+)\s*=\s*\([^)]*\)\s*=>\s*\{\s*return\s+<[^>]+>[\s\S]*?\}')
CONCISE_ARROW_FUNCTION_COMPONENT_REGEX = re.compile(r'const\s+(\w+)\s*=\s*\([^)]*\)\s*=>\s*<[^>]+>')


In [16]:
def find_components(file_path, content):
    """
    Finds React components in the content of a JavaScript file.

    Args:
        file_path (str): The path of the file being scanned.
        content (str): The content of the JavaScript file.

    Returns:
        list: A list of tuples with the component type and name.
    """
    components = []

    # Match class-based components
    for match in CLASS_COMPONENT_REGEX.finditer(content):
        components.append(("Class Component", match.group(1), file_path))
    
    # Match function declarations
    for match in FUNCTION_COMPONENT_REGEX.finditer(content):
        components.append(("Function Component", match.group(1), file_path))
    
    # Match arrow functions with return
    for match in ARROW_FUNCTION_COMPONENT_REGEX.finditer(content):
        components.append(("Arrow Function Component", match.group(1), file_path))
    
    # Match concise arrow functions
    for match in CONCISE_ARROW_FUNCTION_COMPONENT_REGEX.finditer(content):
        components.append(("Concise Arrow Function Component", match.group(1), file_path))
    
    return components


In [17]:
# Test the find_components function
test_content = """
class MyClass extends React.Component {
  render() {
    return <div>Hello</div>;
  }
}

function MyFunction() {
  return <div>Hello</div>;
}

const MyArrowFunction = () => {
  return <div>Hello</div>;
}

const MyConciseArrowFunction = () => <div>Hello</div>;
"""

test_path = "test_file.js"
components = find_components(test_path, test_content)

# Display the results
for component in components:
    print(f"{component[0]}: {component[1]} found in {component[2]}")


Class Component: MyClass found in test_file.js
Function Component: MyFunction found in test_file.js
Arrow Function Component: MyArrowFunction found in test_file.js
Concise Arrow Function Component: MyConciseArrowFunction found in test_file.js


In [18]:
def scan_directory(directory):
    """
    Scans a directory recursively for JavaScript files and identifies React components.

    Args:
        directory (str): The directory to scan.

    Returns:
        list: A list of tuples with the component type, name, and file path.
    """
    all_components = []
    
    for root, _, files in os.walk(directory):
        for file in files:
            if file.endswith('.js'):
                file_path = os.path.join(root, file)
                with open(file_path, 'r', encoding='utf-8') as f:
                    content = f.read()
                    all_components.extend(find_components(file_path, content))
    
    return all_components


In [19]:
# Create a test directory with sample files
import tempfile

# Create a temporary directory
test_dir = tempfile.TemporaryDirectory()

# Create sample JavaScript files
sample_files = {
    "file1.js": """
    class SampleClass extends React.Component {
      render() {
        return <div>Hello from SampleClass</div>;
      }
    }
    """,
    "file2.js": """
    function SampleFunction() {
      return <div>Hello from SampleFunction</div>;
    }
    const SampleArrowFunction = () => {
      return <div>Hello from SampleArrowFunction</div>;
    }
    const SampleConciseArrowFunction = () => <div>Hello from SampleConciseArrowFunction</div>;
    """
}

# Write sample files to the test directory
for filename, content in sample_files.items():
    with open(os.path.join(test_dir.name, filename), 'w') as f:
        f.write(content)

# Test the scan_directory function
components = scan_directory(test_dir.name)

# Display the results
for component in components:
    print(f"{component[0]}: {component[1]} found in {component[2]}")

# Clean up the temporary directory
test_dir.cleanup()


Class Component: SampleClass found in C:\Users\FRANCE~1\AppData\Local\Temp\tmpd_ke5pcl\file1.js
Function Component: SampleFunction found in C:\Users\FRANCE~1\AppData\Local\Temp\tmpd_ke5pcl\file2.js
Arrow Function Component: SampleArrowFunction found in C:\Users\FRANCE~1\AppData\Local\Temp\tmpd_ke5pcl\file2.js
Concise Arrow Function Component: SampleConciseArrowFunction found in C:\Users\FRANCE~1\AppData\Local\Temp\tmpd_ke5pcl\file2.js


In [20]:
# Define the directory to scan
base_path = os.getcwd()  # Get the current working directory
react_folder_path = os.path.join(base_path, "react")  # Adjust folder name if different

# Scan the React folder
print(f"Scanning directory: {react_folder_path}")
react_components = scan_directory(react_folder_path)

# Display the results
for component in react_components:
    print(f"{component[0]}: {component[1]} found in {component[2]}")


Scanning directory: C:\Users\Francesco\PycharmProjects\FSS Software Evolution\FSS Software Evolution\react
Concise Arrow Function Component: ErrorView found in C:\Users\Francesco\PycharmProjects\FSS Software Evolution\FSS Software Evolution\react\compiler\crates\react_hermes_parser\tests\fixtures\arrow-function-expr-gating-test.js
Function Component: NoForget found in C:\Users\Francesco\PycharmProjects\FSS Software Evolution\FSS Software Evolution\react\compiler\crates\react_hermes_parser\tests\fixtures\codegen-instrument-forget-gating-test.js
Function Component: NoForget found in C:\Users\Francesco\PycharmProjects\FSS Software Evolution\FSS Software Evolution\react\compiler\crates\react_hermes_parser\tests\fixtures\codegen-instrument-forget-test.js
Arrow Function Component: getJSX found in C:\Users\Francesco\PycharmProjects\FSS Software Evolution\FSS Software Evolution\react\compiler\crates\react_hermes_parser\tests\fixtures\const-propagation-into-function-expression-global.js
Functio

In [21]:
import csv

def save_results_to_file(results, output_file):
    """
    Saves the list of React components to a CSV file.

    Args:
        results (list): A list of tuples containing component type, name, and file path.
        output_file (str): The path to the CSV file for saving results.
    """
    with open(output_file, 'w', newline='', encoding='utf-8') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["Component Type", "Component Name", "File Path"])
        writer.writerows(results)
    print(f"Results saved to {output_file}")


In [22]:
# Define the directory to scan
base_path = os.getcwd()  # Get the current working directory
task1data_path = os.path.join(base_path, "Task1Data")  
os.makedirs(task1data_path, exist_ok=True)  # Create Task1Data folder if it doesn't exist

# Define the React folder path and output file path
react_folder_path = os.path.join(base_path, "react")  # Adjust folder name if different
output_file = os.path.join(task1data_path, "output_react_components(part1).csv")  # Output CSV file

# Step 1: Scan the React folder
print(f"Scanning directory: {react_folder_path}")
react_components = scan_directory(react_folder_path)

# Step 2: Save the results to a CSV file
save_results_to_file(react_components, output_file)


Scanning directory: C:\Users\Francesco\PycharmProjects\FSS Software Evolution\FSS Software Evolution\react
Results saved to C:\Users\Francesco\PycharmProjects\FSS Software Evolution\FSS Software Evolution\Task1Data\output_react_components(part1).csv
